# P04 · Pumps · BEATs embeddings (run on Colab)

**What this notebook does**
1. Downloads the official BEATs code and its pre-trained checkpoint: BEATs iter3+, AudioSet-2M, self-supervised, **not fine-tuned**.
2. Downloads two pump datasets from Zenodo and checks each file against its published MD5:
   - MIMII, 6 dB, pump (7.7 GB);
   - MIMII DUE, development pump data (1.1 GB).
3. Turns every clip into BEATs embeddings. The model is frozen: no training, no fine-tuning.
4. Saves two files to Google Drive › MyDrive › `P04_BEATs`:
   - `mimii_6dB_pump_beats.npz`;
   - `mimii_due_dev_pump_beats.npz`.

**It computes no scores.** Scoring follows `results/mimii_beats/RULES_written_before_running.md`, which was written before this notebook ran. The DUE file stays sealed until its own rules are written.

**Before you run it:** Runtime › Change runtime type › **T4 GPU**. Then Runtime › Run all. Google Drive will ask for permission once.

**Licences:**
- BEATs code and weights: MIT (Microsoft).
- MIMII: CC BY-SA 4.0 (Hitachi).
- MIMII DUE: CC BY-NC-SA 4.0 (Hitachi).

In [ ]:
# 1 · Setup: where files go, which device runs the model
import os, sys, json, time, hashlib, struct, zipfile, base64, subprocess, shutil, urllib.request, re
from pathlib import Path
from collections import Counter
import numpy as np
import torch

TEST = os.environ.get("P04_TEST") == "1"          # only for checking this notebook outside Colab
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, WORK = Path("/content/drive/MyDrive/P04_BEATs"), Path("/content/work")
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_BEATs")), Path(os.environ.get("P04_WORK", "work"))
OUT.mkdir(parents=True, exist_ok=True); WORK.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(no GPU: this will be slow)")
print("torch", torch.__version__, "· outputs go to", OUT)


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    """wget when it exists (shows progress, resumes), otherwise urllib. A failed download leaves no file behind."""
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()

In [ ]:
# 2 · The official BEATs code (github.com/microsoft/unilm/tree/master/beats)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {   # as downloaded on 2026-11-06; a different hash means the upstream code has changed since
    "BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
    "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
    "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424",
}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as on 2026-11-06" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi                    # BEATs computes its filter banks with it (present in Colab)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The pre-trained checkpoint: BEATs iter3+ (AS2M), not fine-tuned (the authors' link, else Microsoft's Azure copy)
CKPT_NAME = "BEATs_iter3_plus_AS2M.pt"
CKPT_SHARE = "https://1drv.ms/u/s!AqeByhGUtINrgcpke6_lRSZEKD5j2Q?e=A3FpOf"   # the link on the authors' page
# The same file on Microsoft's own Azure storage: the link Salesforce's LAVIS library uses (lavis/models/beats_encoder.py)
CKPT_AZURE = ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3_plus_AS2M.pt?sv=2020-08-04"
              "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")


def onedrive_direct(share_url):
    """OneDrive share link → direct download (the documented 'shares' form of the OneDrive API)."""
    b64 = base64.b64encode(share_url.encode()).decode().rstrip("=").replace("/", "_").replace("+", "-")
    return f"https://api.onedrive.com/v1.0/shares/u!{b64}/root/content"


ckpt, ckpt_src = OUT / CKPT_NAME, OUT / (CKPT_NAME + ".source.txt")    # kept in Drive: a second run skips the download
if ckpt.exists() and ckpt.stat().st_size < 10_000_000:
    ckpt.unlink()                           # an error page saved by a failed attempt, not the model
if not ckpt.exists() and not TEST:
    for label, url in (("OneDrive, the authors' link", onedrive_direct(CKPT_SHARE)),
                       ("Microsoft Azure copy, the link LAVIS uses", CKPT_AZURE)):
        try:
            download(url, ckpt)
        except Exception as e:
            print(f"{label}: failed ({e.__class__.__name__})")
        if ckpt.exists() and ckpt.stat().st_size < 10_000_000:
            ckpt.unlink()
        if ckpt.exists():
            ckpt_src.write_text(label); print(f"{label}: downloaded"); break
if not ckpt.exists():
    raise SystemExit(f"Both automatic downloads failed. By hand: open {CKPT_SHARE} in your browser, press Download, "
                     f"put {CKPT_NAME} into Google Drive › MyDrive › P04_BEATs, then run this cell again.")
CKPT_SOURCE = ckpt_src.read_text() if ckpt_src.exists() else "placed in Drive by hand"
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, f"{ckpt.stat().st_size / 1e6:.1f} MB · from: {CKPT_SOURCE} · SHA-256 {CKPT_SHA}")
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)   # only Microsoft-hosted copies or your own download reach this
cfg = BEATsConfig(chk["cfg"])
assert not cfg.finetuned_model, "this checkpoint is fine-tuned; the rules use the pre-trained one"
model = BEATs(cfg)
model.load_state_dict(chk["model"])
model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· width:", cfg.encoder_embed_dim, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Data: download from Zenodo and check the published MD5 (a broken or swapped file is worse than none)
DATA = {
    "mimii": ("https://zenodo.org/records/3384388/files/6_dB_pump.zip?download=1", "6_dB_pump.zip", "a09ba6060c10fc09cd4c8770213b0b9f"),
    "due": ("https://zenodo.org/records/4740355/files/dev_data_pump.zip?download=1", "dev_data_pump.zip", "8c7090f0974336404070315a1e1445f4"),
}
if TEST:
    DATA = json.loads(os.environ["P04_TEST_DATA"])


def get_zip(key):
    url, name, want = DATA[key]
    for place in (OUT / name, WORK / name):            # a copy already in Drive is used as it is
        if place.exists():
            got = md5(place)
            if got == want:
                print(name, "· MD5 matches the Zenodo page"); return place
            print(name, "· found, but MD5 differs: downloading again")
    dest = WORK / name
    download(url, dest)
    got = md5(dest)
    print(name, "· MD5 matches the Zenodo page:", got == want)
    assert got == want, f"stop: {name} is not the published file"
    return dest

In [ ]:
# 5 · Read a WAV straight from the zip (plain or WAVE_FORMAT_EXTENSIBLE), keep channel 0
def read_ch0(data):
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:                                   # extensible: the sub-format GUID starts with the real tag
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    x = np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch)[:, 0].astype(np.float32) / 32768.0
    return x, sr, ch, nf

In [ ]:
# 6 · Embeddings: the model's last-layer output, plus every layer's mean and spread over time
layer_out = {}
for k, layer in enumerate(model.encoder.layers):
    layer.register_forward_hook(lambda m, i, o, k=k: layer_out.__setitem__(k, o[0]))   # o[0]: (time, batch, 768)


@torch.no_grad()
def embed(batch):
    """batch: (B, N) clips of equal length in [-1, 1] at 16 kHz → last-layer mean (B, 768) and per-layer mean/std (B, 12, 768)."""
    x = torch.from_numpy(batch).to(DEVICE)
    last, _ = model.extract_features(x)                       # (B, T, 768): the model's own output
    Hs = torch.stack([layer_out[k] for k in range(len(model.encoder.layers))])   # (L, T, B, 768)
    return (last.mean(1).float().cpu().numpy(),
            Hs.mean(1).permute(1, 0, 2).half().cpu().numpy(),
            Hs.std(1).permute(1, 0, 2).half().cpu().numpy(),
            float((Hs[-1].transpose(0, 1).mean(1) - last.mean(1)).abs().max()))


def run_zip(zpath, out_name, zip_md5, batch_size=16):
    zf = zipfile.ZipFile(zpath)
    names = sorted(n for n in zf.namelist() if n.lower().endswith(".wav"))
    t0, fn, ns, chs, LAST, LMEAN, LSTD, gap = time.time(), [], [], [], [], [], [], 0.0
    buf, bnames = [], []

    def flush():
        nonlocal gap
        l, m, s, g = embed(np.stack(buf))
        LAST.append(l); LMEAN.append(m); LSTD.append(s); fn.extend(bnames); gap = max(gap, g)
        buf.clear(); bnames.clear()

    for i, n in enumerate(names):
        y, sr, ch, nf = read_ch0(zf.read(n))
        assert sr == 16000, (n, sr)
        if buf and len(y) != len(buf[0]):                    # equal lengths inside a batch
            flush()
        buf.append(y); bnames.append(n); ns.append(nf); chs.append(ch)
        if len(buf) == batch_size:
            flush()
        if i % 500 == 0:
            print(f"{i}/{len(names)} clips · {time.time() - t0:.0f} s", flush=True)
    if buf:
        flush()
    meta = {"checkpoint": CKPT_NAME, "checkpoint_sha256": CKPT_SHA, "checkpoint_source": CKPT_SOURCE, "code_sha256": code_sha, "zip": Path(zpath).name,
            "zip_md5": zip_md5, "clips": len(fn), "torch": torch.__version__, "device": DEVICE,
            "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "runtime_s": round(time.time() - t0, 1),
            "last_vs_layer12_max_abs_diff": gap, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST}
    out = OUT / out_name
    np.savez_compressed(out, fname=np.array(fn), n_samples=np.array(ns), n_channels=np.array(chs),
                        last_mean=np.concatenate(LAST), layer_mean=np.concatenate(LMEAN), layer_std=np.concatenate(LSTD),
                        meta=np.array(json.dumps(meta)))
    print(f"saved {out_name}: {len(fn)} clips in {meta['runtime_s']:.0f} s · MD5 {md5(out)}")
    return fn

In [ ]:
# 6b · Smoke test on two clips of noise: shapes and speed, before the long run
t = time.time()
l, m, s, g = embed(np.random.default_rng(0).uniform(-0.1, 0.1, (2, 160000)).astype(np.float32))
print("last layer:", l.shape, "· per layer:", m.shape, s.shape, f"· {time.time() - t:.2f} s for two 10-second clips · check {g:.1e}")

In [ ]:
# 7 · MIMII 6 dB pump → mimii_6dB_pump_beats.npz
z = get_zip("mimii")
fn = run_zip(z, "mimii_6dB_pump_beats.npz", DATA["mimii"][2])
pump = [re.search(r"(id_\d+)", n).group(1) for n in fn]
cond = [n.replace("\\", "/").split("/")[-2] for n in fn]
print(sorted(Counter(zip(pump, cond)).items()))     # counts only — no scores here

In [ ]:
# 8 · MIMII DUE development pump → mimii_due_dev_pump_beats.npz (sealed: counts only, no scores)
z = get_zip("due")
fn = run_zip(z, "mimii_due_dev_pump_beats.npz", DATA["due"][2])
keys = [tuple(re.search(p, Path(n).name).group(0) if re.search(p, Path(n).name) else "?"
              for p in (r"section_\d+", r"source|target", r"train|test", r"normal|anomaly")) for n in fn]
print(sorted(Counter(keys).items()))

## Send the two files back
In Google Drive › MyDrive › `P04_BEATs`, download:
- `mimii_6dB_pump_beats.npz`
- `mimii_due_dev_pump_beats.npz`

Put them in your project folder, next to the MIMII features (`4. Pump Project › MIMII_features`), and say so in the chat. The checkpoint can stay in Drive.